In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("Travel.csv")

In [3]:
df = df.drop(["CustomerID"], axis=1)

In [4]:
def ismale(s):
    if "Fe" in s:
        return 0
    return 1
df["is_male"] = df["Gender"].apply(ismale)
df = df.drop(["Gender"], axis=1)

In [5]:
df

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome,is_male
0,1,41.0,Self Enquiry,3,6.0,Salaried,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0,0
1,0,49.0,Company Invited,1,14.0,Salaried,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0,1
2,1,37.0,Self Enquiry,1,8.0,Free Lancer,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0,1
3,0,33.0,Company Invited,1,9.0,Salaried,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0,0
4,0,NaN,Self Enquiry,1,8.0,Small Business,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,1,49.0,Self Enquiry,3,9.0,Small Business,3,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,1.0,Manager,26576.0,1
4884,1,28.0,Company Invited,1,31.0,Salaried,4,5.0,Basic,3.0,Single,3.0,1,3,1,2.0,Executive,21212.0,1
4885,1,52.0,Self Enquiry,3,17.0,Salaried,4,4.0,Standard,4.0,Married,7.0,0,1,1,3.0,Senior Manager,31820.0,0
4886,1,19.0,Self Enquiry,3,16.0,Small Business,3,4.0,Basic,3.0,Single,3.0,0,5,0,2.0,Executive,20289.0,1


In [6]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

In [7]:
df.isnull().sum()

prodtaken                     0
age                         226
typeofcontact                25
citytier                      0
durationofpitch             251
occupation                    0
numberofpersonvisiting        0
numberoffollowups            45
productpitched                0
preferredpropertystar        26
maritalstatus                 0
numberoftrips               140
passport                      0
pitchsatisfactionscore        0
owncar                        0
numberofchildrenvisiting     66
designation                   0
monthlyincome               233
is_male                       0
dtype: int64

In [8]:
num_cols = df.select_dtypes(include="number").columns
cat_cols = df.select_dtypes(include="object").columns

C:\Users\HP\AppData\Local\Temp\ipykernel_188360\2242950971.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns


In [9]:
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

In [10]:
df.drop_duplicates(inplace=True)
df.isnull().any()

prodtaken                   False
age                         False
typeofcontact               False
citytier                    False
durationofpitch             False
occupation                  False
numberofpersonvisiting      False
numberoffollowups           False
productpitched              False
preferredpropertystar       False
maritalstatus               False
numberoftrips               False
passport                    False
pitchsatisfactionscore      False
owncar                      False
numberofchildrenvisiting    False
designation                 False
monthlyincome               False
is_male                     False
dtype: bool

In [11]:
y = df["prodtaken"]
X = df[[x for x in df.columns if x!="prodtaken"]]

In [12]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer([
    ("OneHotEncoder", oh_transformer, cat_cols),
    ("StandardScaler", numeric_transformer, [x for x in num_cols if x!="prodtaken"]),
    
])


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, random_state=42, test_size=0.2)

In [13]:
X_train = preprocessor.fit_transform(X_train)

In [14]:
X_test = preprocessor.transform(X_test)

In [17]:
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

rfc = RandomForestClassifier(n_estimators=100)
rfc.fit(X_train, y_train)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_pred = rfc.predict(X_test)


# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Print results
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.9211
Precision: 0.9417
Recall   : 0.6243
F1 Score : 0.7508

Confusion Matrix
[[762   7]
 [ 68 113]]

Classification Report
              precision    recall  f1-score   support

           0       0.92      0.99      0.95       769
           1       0.94      0.62      0.75       181

    accuracy                           0.92       950
   macro avg       0.93      0.81      0.85       950
weighted avg       0.92      0.92      0.91       950



In [22]:
from sklearn.ensemble import AdaBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

abc = AdaBoostClassifier(n_estimators=100)
abc.fit(X_train, y_train)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_pred = abc.predict(X_test)


# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Print results
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.8400
Precision: 0.7377
Recall   : 0.2486
F1 Score : 0.3719

Confusion Matrix
[[753  16]
 [136  45]]

Classification Report
              precision    recall  f1-score   support

           0       0.85      0.98      0.91       769
           1       0.74      0.25      0.37       181

    accuracy                           0.84       950
   macro avg       0.79      0.61      0.64       950
weighted avg       0.83      0.84      0.81       950



In [25]:
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

xbc = XGBClassifier(n_estimators=100)
xbc.fit(X_train, y_train)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_pred = abc.predict(X_test)


# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Print results
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.8400
Precision: 0.7377
Recall   : 0.2486
F1 Score : 0.3719

Confusion Matrix
[[753  16]
 [136  45]]

Classification Report
              precision    recall  f1-score   support

           0       0.85      0.98      0.91       769
           1       0.74      0.25      0.37       181

    accuracy                           0.84       950
   macro avg       0.79      0.61      0.64       950
weighted avg       0.83      0.84      0.81       950



In [23]:
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

gbc = GradientBoostingClassifier(n_estimators=100)
gbc.fit(X_train, y_train)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_pred = abc.predict(X_test)


# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Print results
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.8400
Precision: 0.7377
Recall   : 0.2486
F1 Score : 0.3719

Confusion Matrix
[[753  16]
 [136  45]]

Classification Report
              precision    recall  f1-score   support

           0       0.85      0.98      0.91       769
           1       0.74      0.25      0.37       181

    accuracy                           0.84       950
   macro avg       0.79      0.61      0.64       950
weighted avg       0.83      0.84      0.81       950



In [19]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Create model
rfc = RandomForestClassifier(random_state=42)

# Hyperparameter grid
param_grid = {
    "n_estimators": [100, 200, 300],
    "criterion":["gini", "entropy", "log_loss"],
    #"max_depth": [1,2,3,4,5],
    "max_features": ["sqrt", "log2"]
}

# Grid Search
grid_search = GridSearchCV(
    estimator=rfc,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",   # You can use "f1", "roc_auc", etc.
    n_jobs=-1,
    verbose=2
)

# Train
grid_search.fit(X_train, y_train)

# Best parameters
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation Score:")
print(grid_search.best_score_)


# Predictions
y_pred = grid_search.predict(X_test)

# Probability predictions (needed for ROC-AUC)
y_prob = grid_search.predict_proba(X_test)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

# Print results
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best Parameters:
{'criterion': 'gini', 'max_features': 'sqrt', 'n_estimators': 100}

Best Cross Validation Score:
0.900185493377713
Accuracy : 0.9200
Precision: 0.9412
Recall   : 0.6188
F1 Score : 0.7467
ROC-AUC  : 0.9785

Confusion Matrix
[[762   7]
 [ 69 112]]

Classification Report
              precision    recall  f1-score   support

           0       0.92      0.99      0.95       769
           1       0.94      0.62      0.75       181

    accuracy                           0.92       950
   macro avg       0.93      0.80      0.85       950
weighted avg       0.92      0.92      0.91       950



In [24]:
! pip install xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.8/69.5 MB 3.9 MB/s eta 0:00:18
    --------------------------------------- 1.6/69.5 MB 3.6 MB/s eta 0:00:19
   - -------------------------------------- 2.1/69.5 MB 3.5 MB/s eta 0:00:20
   - -------------------------------------- 2.6/69.5 MB 3.0 MB/s eta 0:00:22
   - -------------------------------------- 2.6/69.5 MB 3.0 MB/s eta 0:00:22
   - -------------------------------------- 3.4/69.5 MB 2.7 MB/s eta 0:00:25
   -- ------------------------------------- 3.9/69.5 MB 2.6 MB/s eta 0:00:25
   -- ------------------------------------- 4.5/69.5 MB 2.6 MB/s eta 0:00:25
   -- ------------------------------------- 5.0/69.5 MB 2.7 MB/s eta 0:00:25
   --- ------------------------------------ 5.5/69.5 MB 2.6 MB/s eta 0:00:25
   --- ------------------------------------ 5.5/69.5 MB 2.6 MB/s eta 0:00:25
   --- ------